# NYC 311 Service Requests — Data Cleaning & Exploratory Analysis

**By Nicholas Osani**

10,000 NYC 311 service requests pulled from the NYC Open Data portal (Socrata API, dataset `erm2-nwe9`) and saved to `311_service_requests.csv`. This notebook takes the raw pull through cleaning and into exploratory analysis of complaint patterns.

## 1. Loading the data

In [1]:
import pandas as pd

# Reload the saved CSV file
df = pd.read_csv("311_service_requests.csv")

print(df.shape)
df.head()

(10000, 47)


,unique_key,created_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,incident_address,street_name,...,resolution_description,resolution_action_updated_date,vehicle_type,facility_type,bridge_highway_name,bridge_highway_segment,taxi_company_borough,bridge_highway_direction,road_ramp,due_date
0,66932951,2025-11-24T02:06:15.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10306.0,1742 RICHMOND ROAD,RICHMOND ROAD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,66927366,2025-11-24T02:05:57.000,NYPD,New York City Police Department,Noise - Residential,Loud Television,Residential Building/House,10002.0,280 MADISON STREET,MADISON STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,66927332,2025-11-24T02:05:27.000,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,Store/Commercial,11103.0,42-12 BROADWAY,BROADWAY,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,66927308,2025-11-24T02:04:27.000,NYPD,New York City Police Department,Illegal Parking,Commercial Overnight Parking,Street/Sidewalk,10470.0,343 EAST 238 STREET,EAST 238 STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,66931824,2025-11-24T02:03:41.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,Street/Sidewalk,11385.0,2025 WOODBINE STREET,WOODBINE STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Cleaning the data

Steps: inspect dtypes, drop exact duplicates, handle missing values, drop unneeded columns, parse dates, and engineer `days_to_close` (closed minus created). Rows still open (no `closed_date`) are excluded from duration analysis — you can't measure resolution time on unresolved cases.

In [2]:
# Check the columns and basic info
print("Columns in the dataset:")
print(list(df.columns))

print("Data types (first few columns):")
print(df.dtypes.head())

Columns in the dataset:
['unique_key', 'created_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'status', 'community_board', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'latitude', 'longitude', 'location', ':@computed_region_efsh_h5xi', ':@computed_region_f5dn_yrer', ':@computed_region_yeji_bk3q', ':@computed_region_92fq_4b7q', ':@computed_region_sbqj_enih', ':@computed_region_7mpf_4k6g', 'taxi_pick_up_location', 'closed_date', 'resolution_description', 'resolution_action_updated_date', 'vehicle_type', 'facility_type', 'bridge_highway_name', 'bridge_highway_segment', 'taxi_company_borough', 'bridge_highway_direction', 'road_ramp', 'due_date']
Data types (first few columns):
unique_key       

In [3]:
# Remove duplicate rows
before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]

print("Number of duplicates removed:", before_rows - after_rows)

Number of duplicates removed: 4


In [4]:
# Check missing values for key columns (only if they exist)
if "closed_date" in df.columns:
    print("Missing closed_date values:", df["closed_date"].isnull().sum())

if "complaint_type" in df.columns:
    print("Missing complaint_type values:", df["complaint_type"].isnull().sum())

Missing closed_date values: 3965
Missing complaint_type values: 0


In [5]:
# Drop rows that are missing key values
subset_cols = []

if "closed_date" in df.columns:
    subset_cols.append("closed_date")

if "complaint_type" in df.columns:
    subset_cols.append("complaint_type")

if len(subset_cols) > 0:
    df = df.dropna(subset=subset_cols)

print("Shape after dropping rows with missing key values:", df.shape)

Shape after dropping rows with missing key values: (6031, 47)


In [6]:
# Drop some columns that are not needed for our analysis
drop_candidates = [
    "location_state",
    "facility_type",
    "intersection_street_1",
    "intersection_street_2",
    "city",
    "location_zip",
    "park_borough",
    "latitude",
    "longitude",
    "road_ramp",
    "bbl",
    "location_city",
    "open_data_channel_type",
    "cross_street_1",
    "cross_street_2",
    "bridge_highway_direction",
    "bridge_highway_segment",
    "x_coordinate_state_plane",
    "y_coordinate_state_plane",
    "taxi_company_borough",
    "taxi_pick_up_location",
    "vehicle_type",
    "geometry"  
]

for col in drop_candidates:
    if col in df.columns:
        df = df.drop([col], axis=1)

print("Shape after dropping unnecessary columns:", df.shape)
print("Remaining columns:")
print(list(df.columns))

Shape after dropping unnecessary columns: (6031, 28)
Remaining columns:
['unique_key', 'created_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'address_type', 'landmark', 'status', 'community_board', 'borough', 'park_facility_name', 'location', ':@computed_region_efsh_h5xi', ':@computed_region_f5dn_yrer', ':@computed_region_yeji_bk3q', ':@computed_region_92fq_4b7q', ':@computed_region_sbqj_enih', ':@computed_region_7mpf_4k6g', 'closed_date', 'resolution_description', 'resolution_action_updated_date', 'bridge_highway_name', 'due_date']


In [7]:
# Convert date strings to datetime objects
if "created_date" in df.columns:
    df["created_date"] = pd.to_datetime(df["created_date"])

if "closed_date" in df.columns:
    df["closed_date"] = pd.to_datetime(df["closed_date"])

print(df[["created_date", "closed_date"]].head())
print(df[["created_date", "closed_date"]].dtypes)

          created_date         closed_date
13 2025-11-24 01:55:19 2025-11-24 02:02:17
14 2025-11-24 01:55:15 2025-11-24 01:59:06
21 2025-11-24 01:51:28 2025-11-24 01:52:21
49 2025-11-24 01:41:31 2025-11-24 01:57:41
53 2025-11-24 01:40:27 2025-11-24 01:57:43
created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object


In [8]:
# Calculate days to close (closed_date - created_date)
if "created_date" in df.columns and "closed_date" in df.columns:
    df["days_to_close"] = (df["closed_date"] - df["created_date"]).dt.days

# Extract year and month for temporal analysis
if "created_date" in df.columns:
    df["year"] = df["created_date"].dt.year
    df["month_num"] = df["created_date"].dt.month

print(df[["created_date", "closed_date", "days_to_close", "year", "month_num"]].head())

          created_date         closed_date  days_to_close  year  month_num
13 2025-11-24 01:55:19 2025-11-24 02:02:17              0  2025         11
14 2025-11-24 01:55:15 2025-11-24 01:59:06              0  2025         11
21 2025-11-24 01:51:28 2025-11-24 01:52:21              0  2025         11
49 2025-11-24 01:41:31 2025-11-24 01:57:41              0  2025         11
53 2025-11-24 01:40:27 2025-11-24 01:57:43              0  2025         11


In [9]:
print("Final shape of cleaned DataFrame:", df.shape)
print("Sample of cleaned data:")
df.head()

Final shape of cleaned DataFrame: (6031, 31)
Sample of cleaned data:


,unique_key,created_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,incident_address,street_name,...,:@computed_region_sbqj_enih,:@computed_region_7mpf_4k6g,closed_date,resolution_description,resolution_action_updated_date,bridge_highway_name,due_date,days_to_close,year,month_num
13,66928049,2025-11-24 01:55:19,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11433.0,108-26 159 STREET,159 STREET,...,61.0,61.0,2025-11-24 02:02:17,The New York City Police Department responded ...,2025-11-24T02:02:26.000,NaN,NaN,0,2025,11
14,66933632,2025-11-24 01:55:15,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11237.0,234 STARR STREET,STARR STREET,...,53.0,53.0,2025-11-24 01:59:06,The New York City Police Department responded ...,2025-11-24T01:59:12.000,NaN,NaN,0,2025,11
21,66929136,2025-11-24 01:51:28,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11237.0,1480 JEFFERSON AVENUE,JEFFERSON AVENUE,...,53.0,53.0,2025-11-24 01:52:21,The New York City Police Department responded ...,2025-11-24T01:52:25.000,NaN,NaN,0,2025,11
49,66930180,2025-11-24 01:41:31,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10452.0,111 TUDOR PLACE,TUDOR PLACE,...,27.0,27.0,2025-11-24 01:57:41,The New York City Police Department responded ...,2025-11-24T01:57:50.000,NaN,NaN,0,2025,11
53,66933528,2025-11-24 01:40:27,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10452.0,111 TUDOR PLACE,TUDOR PLACE,...,27.0,27.0,2025-11-24 01:57:43,The New York City Police Department responded ...,2025-11-24T01:57:51.000,NaN,NaN,0,2025,11


## 3. Analyzing the data

Categorical summaries of the cleaned data: complaint mix, agency workload, and geographic/temporal patterns.

In [10]:
#summary statistics for 4 parameters
df.describe()
columns = ['agency', 'complaint_type', 'descriptor', 'location_type']
print(df[columns].describe())

       agency       complaint_type        descriptor    location_type
count    6031                 6031              5883             5776
unique      9                   50               124               21
top      NYPD  Noise - Residential  Loud Music/Party  Street/Sidewalk
freq     5360                 2062              1858             2797


In [11]:
# Top 10 most common complaint types
if "complaint_type" in df.columns:
    top_complaints = df["complaint_type"].value_counts().head(10)
    print("Top 10 complaint types:")
    print(top_complaints)

Top 10 complaint types:
complaint_type
Noise - Residential            2062
Illegal Parking                1476
Blocked Driveway                551
Noise - Street/Sidewalk         392
Noise - Commercial              327
HEAT/HOT WATER                  223
Noise - Vehicle                 117
Abandoned Vehicle               117
Encampment                       90
Non-Emergency Police Matter      73
Name: count, dtype: int64


In [12]:
#Agency complaints ammount descending
if "agency" in df.columns:
    agency_counts = df["agency"].value_counts()
    print("Complaint counts by agency:")
    print(agency_counts)

Complaint counts by agency:
agency
NYPD     5360
HPD       229
DOB       157
DSNY       80
DHS        79
DOT        60
DEP        37
DPR        24
DOHMH       5
Name: count, dtype: int64


In [13]:
#311 complaints this year
if "year" in df.columns:
    complaints_by_year = df["year"].value_counts().sort_index()
    print("Counts of 311 complaints by year:")
    print(complaints_by_year)

Counts of 311 complaints by year:
year
2025    6031
Name: count, dtype: int64


In [14]:
#Complaint number during specific hour of the day
df['hour'] = df['created_date'].dt.hour
df.groupby('hour').size()

hour
0     492
1     332
2     213
3     172
4     129
5     119
6     105
7     151
8     195
9     245
10    225
11    286
12    329
13    318
14    291
15    231
16    187
17    193
18    187
19    226
20    275
21    252
22    279
23    599
dtype: int64

In [15]:
# Complaint counts by borough
if "borough" in df.columns:
    complaints_by_borough = df["borough"].value_counts()
    print("Complaints by borough:")
    print(complaints_by_borough)

Complaints by borough:
borough
BROOKLYN         1645
BRONX            1527
QUEENS           1516
MANHATTAN        1207
STATEN ISLAND     135
Unspecified         1
Name: count, dtype: int64


## Key findings

- **10,000 raw rows → 6,031** after cleaning: 4 exact duplicates removed; ~3,965 rows dropped for missing `closed_date` (still-open cases).
- **Noise dominates**: `Noise - Residential` is the top complaint type (2,062), followed by `Illegal Parking` (1,476) and `Blocked Driveway` (551) — 50 distinct complaint types total.
- **NYPD carries the load**: 5,360 of 6,031 cleaned complaints (89%) route to NYPD.
- **Brooklyn leads** boroughs with 1,645 complaints; Staten Island has just 135.